# Step 2: Data Cleaning with Pandas
Load raw CSVs, fix issues, validate referential integrity, export cleaned CSVs.

In [1]:
import pandas as pd
from datetime import datetime

customers_df = pd.read_csv('../data/raw/customers.csv')
products_df = pd.read_csv('../data/raw/products.csv')
orders_df = pd.read_csv('../data/raw/orders.csv')
order_items_df = pd.read_csv('../data/raw/order_items.csv')

print(customers_df.shape, products_df.shape, orders_df.shape, order_items_df.shape)


(206, 5) (60, 4) (810, 4) (2073, 5)


## Clean customers
Drop duplicates, drop rows with missing name/email.

In [2]:
customers_df = customers_df.drop_duplicates(subset='customer_id')
customers_df = customers_df.dropna(subset=['name', 'email'])
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'], errors='coerce')
customers_df = customers_df.dropna(subset=['signup_date'])
customers_df.shape


(185, 5)

## Clean products
Fix negative prices, fill missing category.

In [3]:
products_df['price'] = products_df['price'].abs()
products_df['category'] = products_df['category'].fillna('Unknown')
products_df = products_df.drop_duplicates(subset='product_id')
products_df.shape


(60, 4)

## Clean orders
Drop duplicates, remove future dates, remove orders with customer_id not in customers table, fill missing status.

In [4]:
orders_df = orders_df.drop_duplicates(subset='order_id')
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], errors='coerce')
orders_df = orders_df.dropna(subset=['order_date'])

today = pd.Timestamp(datetime.today().date())
orders_df = orders_df[orders_df['order_date'] <= today]

orders_df['status'] = orders_df['status'].fillna('unknown')

valid_customer_ids = set(customers_df['customer_id'])
orders_df = orders_df[orders_df['customer_id'].isin(valid_customer_ids)]

orders_df.shape


(728, 4)

## Clean order_items
Remove rows with order_id not in orders, fix negative quantity, drop missing unit_price.

In [5]:
valid_order_ids = set(orders_df['order_id'])
order_items_df = order_items_df[order_items_df['order_id'].isin(valid_order_ids)]

order_items_df['quantity'] = order_items_df['quantity'].abs()
order_items_df = order_items_df.dropna(subset=['unit_price'])

valid_product_ids = set(products_df['product_id'])
order_items_df = order_items_df[order_items_df['product_id'].isin(valid_product_ids)]

order_items_df.shape


(1874, 5)

## Referential integrity check

In [6]:
print('orders with bad customer_id:', (~orders_df['customer_id'].isin(valid_customer_ids)).sum())
print('order_items with bad order_id:', (~order_items_df['order_id'].isin(valid_order_ids)).sum())
print('order_items with bad product_id:', (~order_items_df['product_id'].isin(valid_product_ids)).sum())


orders with bad customer_id: 0
order_items with bad order_id: 0
order_items with bad product_id: 0


## Save cleaned CSVs

In [7]:
customers_df.to_csv('../data/cleaned/customers_clean.csv', index=False)
products_df.to_csv('../data/cleaned/products_clean.csv', index=False)
orders_df.to_csv('../data/cleaned/orders_clean.csv', index=False)
order_items_df.to_csv('../data/cleaned/order_items_clean.csv', index=False)

print('customers:', len(customers_df))
print('products:', len(products_df))
print('orders:', len(orders_df))
print('order_items:', len(order_items_df))


customers: 185
products: 60
orders: 728
order_items: 1874
